In [1]:
import torch, pandas as pd, numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

OUT = "../data/prepared"
catalogue = pd.read_csv(f"{OUT}/catalogue.csv")
ASINS = list(catalogue["asin"])          # fixed order of the 15 products

class BinDataset(Dataset):
    def __init__(self, group, train=False):
        self.group = group
        self.labels = pd.read_csv(f"{OUT}/{group}/labels.csv", index_col=0)[ASINS]
        steps = [transforms.ToTensor(),
                 transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]
        if train:   # small colour changes so the model doesn't memorise
            steps.insert(0, transforms.ColorJitter(brightness=0.2, contrast=0.2))
        self.tf = transforms.Compose(steps)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        bin_id = self.labels.index[i]
        img = Image.open(f"{OUT}/{self.group}/{bin_id}").convert("RGB")
        counts = torch.tensor(self.labels.iloc[i].values, dtype=torch.float32)
        return self.tf(img), counts

train_dl = DataLoader(BinDataset("train", train=True), batch_size=32, shuffle=True)
val_dl   = DataLoader(BinDataset("validation"), batch_size=32)

x, y = next(iter(train_dl))
print("images:", x.shape)    # expect [32, 3, 224, 224]
print("answers:", y.shape)   # expect [32, 15]
print("first answer row:", y[0])

images: torch.Size([32, 3, 224, 224])
answers: torch.Size([32, 15])
first answer row: tensor([0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])


In [2]:
import torch.nn as nn
from torchvision import models

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using:", device)

# Pretrained ResNet-18, last layer replaced with 15 outputs (one count per product)
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, len(ASINS))
model = model.to(device)

loss_fn = nn.SmoothL1Loss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

def run_epoch(dl, train=True):
    model.train(train)
    total, n = 0.0, 0
    with torch.set_grad_enabled(train):
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = loss_fn(out, y)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total += loss.item() * len(x)
            n += len(x)
    return total / n

# One epoch first, just to check that it runs and how long it takes
import time
t = time.time()
tr = run_epoch(train_dl, train=True)
va = run_epoch(val_dl, train=False)
print(f"train loss {tr:.4f} | val loss {va:.4f} | {time.time()-t:.0f} seconds")

Using: mps
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /Users/praneeth/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:10<00:00, 4.48MB/s]


train loss 0.0735 | val loss 0.0496 | 21 seconds


In [3]:
import os
os.makedirs("../models", exist_ok=True)

def val_metrics(dl):
    model.eval()
    tp = fp = fn = 0
    exact = 0
    total = 0
    with torch.no_grad():
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            pred = model(x).round().clamp(0, 5)
            tp += ((pred > 0) & (y > 0)).sum().item()
            fp += ((pred > 0) & (y == 0)).sum().item()
            fn += ((pred == 0) & (y > 0)).sum().item()
            exact += (pred == y).sum().item()
            total += y.numel()
    p = tp / max(tp + fp, 1)
    r = tp / max(tp + fn, 1)
    f1 = 2 * p * r / max(p + r, 1e-9)
    return p, r, f1, exact / total

EPOCHS = 15
best_f1 = -1
for ep in range(1, EPOCHS + 1):
    tr = run_epoch(train_dl, train=True)
    va = run_epoch(val_dl, train=False)
    p, r, f1, acc = val_metrics(val_dl)
    print(f"epoch {ep:2d} | train {tr:.4f} | val {va:.4f} | "
          f"precision {p:.2f} recall {r:.2f} F1 {f1:.2f} | exact {acc:.3f}")
    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), "../models/resnet18_best.pt")

print("Best validation F1:", round(best_f1, 3))

epoch  1 | train 0.0385 | val 0.0401 | precision 0.78 recall 0.38 F1 0.51 | exact 0.944
epoch  2 | train 0.0264 | val 0.0370 | precision 0.71 recall 0.60 F1 0.65 | exact 0.948
epoch  3 | train 0.0169 | val 0.0356 | precision 0.77 recall 0.57 F1 0.66 | exact 0.951
epoch  4 | train 0.0120 | val 0.0331 | precision 0.79 recall 0.58 F1 0.67 | exact 0.950
epoch  5 | train 0.0092 | val 0.0305 | precision 0.84 recall 0.59 F1 0.69 | exact 0.955
epoch  6 | train 0.0082 | val 0.0303 | precision 0.82 recall 0.64 F1 0.72 | exact 0.957
epoch  7 | train 0.0070 | val 0.0302 | precision 0.86 recall 0.53 F1 0.66 | exact 0.955
epoch  8 | train 0.0079 | val 0.0297 | precision 0.84 recall 0.59 F1 0.70 | exact 0.956
epoch  9 | train 0.0069 | val 0.0282 | precision 0.91 recall 0.58 F1 0.71 | exact 0.960
epoch 10 | train 0.0060 | val 0.0271 | precision 0.91 recall 0.60 F1 0.73 | exact 0.961
epoch 11 | train 0.0059 | val 0.0278 | precision 0.89 recall 0.63 F1 0.74 | exact 0.961
epoch 12 | train 0.0050 | val 0.